# Chapter 4: Storage and  Retrieval
### *Depolama ve Erişim* — Designing Data-Intensive Applications

> *"Hayatın sıkıntılarından biri, herkesin her şeyi biraz yanlış adlandırmasıdır. Bu da dünyadaki her şeyi olması gerekenden biraz daha anlaşılması zor hale getirir. Bir bilgisayar, öncelikli olarak aritmetik anlamda hesap yapmaz. Onlar öncelikle dosya sistemleridir."*
> — Richard Feynman, 1985

---

Bu bölüm şu temel soruyu yanıtlar: **Bir veritabanı veriyi nasıl depolar ve nasıl geri getirir?**

Uygulama geliştiricisi olarak neden bu konuyu bilmeniz gerekiyor? Kendi storage engine'inizi sıfırdan yazmayacaksınız, ama **uygulamanıza uygun storage engine'i seçmek zorunda kalacaksınız**. Bunu yapmak için storage engine'in kapağın altında ne yaptığını kabaca bilmeniz gerekir.

Bu bölümde iki ana storage engine ailesi incelenir:
- **Log-structured** storage engines (immutable dosyalar yazar)
- **B-tree** tabanlı storage engines (veriyi yerinde günceller)

Bölümün ikinci yarısında analytics için **column-oriented storage** ve gelişmiş index yapıları ele alınır.

---
## 1. Storage and Indexing for OLTP
### *OLTP için Depolama ve İndeksleme*

**OLTP (Online Transaction Processing):** Çok sayıda küçük işlemin hızlıca yapıldığı sistemler (örn: e-ticaret, bankacılık). Her işlem küçük sayıda satırı okur/yazar.

### 1.1 Dünyanın En Basit Veritabanı — Append-Only Log

Bölüm, iki bash fonksiyonuyla implement edilmiş dünyaca en basit veritabanıyla başlar:

In [1]:
# Bash'deki orijinal kod Python'a çevrilmiş hali
import csv
import os

DB_FILE = "/tmp/simple_database.txt"

def db_set(key, value):
    """Key-value çiftini dosyanın SONUNA ekler (append-only)."""
    with open(DB_FILE, 'a') as f:
        f.write(f"{key},{value}\n")

def db_get(key):
    """Dosyanın TAMAMINI tarar, key'e sahip son değeri döndürür."""
    last_value = None
    if not os.path.exists(DB_FILE):
        return None
    with open(DB_FILE, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith(f"{key},"):
                last_value = line[len(f"{key},"):]
    return last_value

# Test edelim
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

db_set(12, '{"name":"London","attractions":["Big Ben","London Eye"]}')
db_set(42, '{"name":"San Francisco","attractions":["Golden Gate Bridge"]}')
print("42'nin ilk değeri:", db_get(42))

# Aynı key'i güncelleyelim
db_set(42, '{"name":"San Francisco","attractions":["Exploratorium"]}')
print("42'nin güncel değeri:", db_get(42))

# Dosyanın içine bakalım — eski değer hâlâ orada!
print("\nDosyanın tüm içeriği:")
with open(DB_FILE) as f:
    print(f.read())

42'nin ilk değeri: {"name":"San Francisco","attractions":["Golden Gate Bridge"]}
42'nin güncel değeri: {"name":"San Francisco","attractions":["Exploratorium"]}

Dosyanın tüm içeriği:
12,{"name":"London","attractions":["Big Ben","London Eye"]}
42,{"name":"San Francisco","attractions":["Golden Gate Bridge"]}
42,{"name":"San Francisco","attractions":["Exploratorium"]}



#### Nasıl Çalışır?

- **Storage formatı:** Her satır `key,value` biçiminde bir text dosyasına yazılır (CSV gibi).
- `db_set` her çağrıldığında dosyanın **sonuna yazar** (`append`). Eski değer silinmez!
- `db_get` ise dosyayı **baştan sona tarar** ve aynı key'e ait son satırı döndürür (`tail -n 1` mantığı).

#### Neden Append-Only?

Bir dosyanın sonuna ekleme yapmak (**append**) disk üzerinde son derece verimli bir işlemdir çünkü disk kafasının hareket etmesi gerekmez. Gerçek veritabanları da içsel olarak **log** (eklemeli veri dosyası) kullanır.

> ⚠️ **Önemli Not:** Bu bağlamda "log" kelimesi uygulama günlükleri (application logs) anlamına **gelmez**. Burada log, diske yazılan **append-only kayıt dizisi** anlamındadır. Binary olabilir, insan tarafından okunabilir olmak zorunda değildir.

#### Performans Sorunu: O(n) Lookup

- `db_set` → **O(1)** - Çok hızlı, dosyanın sonuna ekle.
- `db_get` → **O(n)** - Çok yavaş! Veritabanında n kayıt varsa, bir arama n adım sürer. 2x kayıt = 2x daha yavaş.

Milyonlarca kayıt olunca bu yöntem tamamen işe yaramaz hale gelir. Çözüm: **Index**.

### 1.2 Index (İndeks)

Belirli bir key'in değerini veritabanında verimli bulmak için farklı bir veri yapısına ihtiyacımız var: **index**.

- Index, asıl veriden **türetilmiş ek bir yapıdır**.
- Veritabanları index eklemenize/kaldırmanıza izin verir; bu işlem **veritabanının içeriğini etkilemez**, yalnızca sorgu performansını etkiler.
- **Write/Read Trade-off:** Her index, yazma işlemlerini yavaşlatır (index de güncellenmelidir). Ancak okuma işlemlerini büyük ölçüde hızlandırır.
- Bu nedenle veritabanları her şeyi otomatik olarak index'lemez. Hangi kolonların index'leneceğine **uygulama geliştiricisi veya DBA** karar verir.

> **Altın Kural:** İyi seçilmiş index'ler okuma sorgularını hızlandırır, ama her index ekstra disk alanı tüketir ve yazmaları yavaşlatır.

---
## 2. Log-Structured Storage
### *Log-Yapılı Depolama*

Bu yaklaşım, append-only dosyayı temel alır ama okuma performansını iyileştirmek için çeşitli mekanizmalar ekler.

### 2.1 Hash Table Index (In-Memory Hash Map)

En basit index yaklaşımı: Her key'i, log dosyasındaki **byte offset**'iyle eşleştiren bir **in-memory hash map** tutmak.

**Nasıl Çalışır?**
1. `db_set` çağrıldığında veri dosyanın sonuna yazılır.
2. Aynı anda hash map şöyle güncellenir: `hash_map[key] = byte_offset_of_new_write`
3. `db_get` çağrıldığında hash map'ten offset alınır, dosyada o konuma `seek` yapılır ve değer okunur.
4. Veri filesystem cache'indeyse disk I/O bile yapılmaz!

Bu aslında **Bitcask** (Riak'ın default storage engine'i) yaklaşımıdır.

In [2]:
# Hash index yaklaşımının Python simülasyonu

class SimpleHashIndexDB:
    def __init__(self, filepath):
        self.filepath = filepath
        self.index = {}  # key -> byte offset in file
        
        # Dosyayı sıfırla
        open(filepath, 'wb').close()
    
    def set(self, key, value):
        entry = f"{key},{value}\n".encode()
        with open(self.filepath, 'ab') as f:
            offset = f.tell()       # Mevcut konum = yeni entry'nin offset'i
            f.write(entry)
        self.index[key] = offset    # Hash map güncellenir
    
    def get(self, key):
        if key not in self.index:
            return None
        offset = self.index[key]    # O(1) lookup!
        with open(self.filepath, 'rb') as f:
            f.seek(offset)          # Doğrudan offset'e git
            line = f.readline().decode().strip()
            return line.split(',', 1)[1]

db = SimpleHashIndexDB("/tmp/hash_index_db.bin")
db.set("user:1", '{"name": "Alice", "age": 30}')
db.set("user:2", '{"name": "Bob", "age": 25}')
db.set("user:1", '{"name": "Alice", "age": 31}')  # Güncelleme

print("user:1 =>", db.get("user:1"))   # En son değer
print("user:2 =>", db.get("user:2"))
print("\nIndex map:", db.index)

user:1 => {"name": "Alice", "age": 31}
user:2 => {"name": "Bob", "age": 25}

Index map: {'user:1': 70, 'user:2': 36}


#### Hash Table Index'in Sorunları

Bu yaklaşımın birkaç önemli sınırı vardır:

1. **Disk alanı bitmesi:** Eski log entry'leri silinmez; yazma devam ederse disk dolabilir.
2. **Hash map kalıcı değildir:** Veritabanı yeniden başlatıldığında hash map bellekten silinir. Yeniden oluşturmak için tüm log dosyası taranmalıdır — çok yavaş olabilir.
3. **Hash table bellekte olmalıdır:** Disk üzerinde hash table tutmak çok verimsizdir (random I/O, büyüdüğünde yavaşlar, hash collision karmaşıklığı).
4. **Range query desteklenmez:** `key 10000` ile `19999` arasındaki tüm değerlere erişmek isteseydiniz, her birini tek tek aramak zorunda kalırsınız. Hash table sıralı değildir.

Bu sorunları çözmek için çok daha iyi bir format geliştirilmiştir: **SSTable**.

---
## 3. SSTable (Sorted String Table) ve LSM-Tree
### *Sıralı String Tablosu ve Log-Yapılı Birleştirme Ağacı*

### 3.1 SSTable Dosya Formatı

**SSTable (Sorted String Table):** Key-value çiftlerini **key'e göre sıralı** olarak saklayan bir dosya formatıdır. Her key dosyada yalnızca bir kez yer alır.

Hash table index ile karşılaştırma:

| Özellik | Hash Index | SSTable |
|---------|------------|--------|
| Key sıralaması | Yok | Sıralı |
| Tüm key'ler bellekte mi? | Evet | Hayır (sparse index) |
| Range query | ❌ | ✅ |
| Compression | Zor | Kolay |

#### Sparse Index (Seyrek İndeks)

SSTable'da **tüm key'leri bellekte tutmaya gerek yoktur**. Bunun yerine:
- Key-value çiftleri birkaç kilobaytlık **block**'lara gruplandırılır.
- Her block'un **ilk key'i** bir **sparse index**'te saklanır.
- Arama yaparken:
  1. Sparse index'te hangi iki key arasında kaldığı bulunur.
  2. Disk'te o bloğa `seek` yapılır ve blok taranır.

**Örnek:** Sparse index'te `handbag` ve `handsome` key'leri var. `handiwork` aranıyor:
- Sıralama nedeniyle `handiwork` mutlaka `handbag` ile `handsome` arasındadır.
- `handbag`'in offset'ine git, oradan küçük bir blok tara.

Her block ayrıca **sıkıştırılabilir** (compressed) — bu hem disk alanı hem de I/O band genişliği tasarrufu sağlar.

### 3.2 SSTable'ların Oluşturulması ve Birleştirilmesi

SSTable okuma açısından mükemmel, ama yazma açısından zorludur. Yeni bir key geldiğinde ortaya sıralı olarak ekleyemezsiniz. Tüm dosyayı yeniden yazmak çok pahalıdır.

Çözüm: **Log-structured yaklaşım** — append-only log ile sıralı dosyanın hibriti:

#### Adım Adım Algoritma:

**1. Memtable (In-Memory Buffer)**
- Yazma geldiğinde önce bellekteki **memtable**'a eklenir.
- Memtable bir **red-black tree**, **skip list** veya **trie** gibi sıralı in-memory veri yapısıdır.
- Bu yapılar sayesinde key'ler herhangi bir sırada eklenebilir, verimli arama yapılabilir ve sıralı çıktı alınabilir.

**2. SSTable'a Flush**
- Memtable birkaç megabayt büyüklüğüne ulaştığında sıralı biçimde diske **SSTable dosyası olarak yazılır**.
- Bu yeni SSTable, veritabanının en son **segmenti** olur.
- Eski segment dosyaları birer birer yanında durur, her birinin kendi sparse index'i vardır.
- Yeni SSTable yazılırken yazma işlemleri yeni bir memtable örneğine devam eder.

**3. Okuma Sırası**
- Okuma geldiğinde önce **memtable** kontrol edilir.
- Bulunamazsa **en yeni SSTable segment**'ine bakılır.
- Hâlâ bulunamazsa sırasıyla **daha eski segment**'lere bakılır.
- Hiçbirinde yoksa key veritabanında yoktur.

**4. Compaction (Sıkıştırma/Birleştirme)**
- Arka planda **merging ve compaction** işlemi yapılır: Segment dosyaları birleştirilir, üzerine yazılmış veya silinmiş değerler temizlenir.

**Crash Recovery için WAL:**
- Memtable bellekte olduğu için çökme durumunda kaybolur.
- Bunun önlemi: Her yazma, ayrı bir **write-ahead log (WAL)** dosyasına anında eklenir.
- Bu log sıralı değildir; amacı yalnızca çökmeden sonra memtable'ı yeniden oluşturmaktır.
- Memtable SSTable'a flush edildiğinde bu log'un ilgili kısmı silinir.

In [3]:
# SSTable merge işlemi — mergesort mantığıyla
# Birden fazla sıralı segment dosyasını tek sıralı çıktıda birleştir

import heapq

def merge_sstables(*segments):
    """
    Birden fazla sıralı SSTable segmentini birleştir.
    Aynı key için en yeni segmentin değerini al.
    segment[0] en yeni, segment[-1] en eski.
    """
    # Her segment'i sıralı iterator olarak wrap et, öncelik: yeni -> eski
    # (key, segment_index, value) -> küçük key önce gelir
    heap = []
    iterators = [iter(seg) for seg in segments]
    
    for i, it in enumerate(iterators):
        try:
            k, v = next(it)
            heapq.heappush(heap, (k, i, v, it))
        except StopIteration:
            pass
    
    result = {}
    last_seen_key = None
    
    while heap:
        key, seg_idx, value, it = heapq.heappop(heap)
        
        # Aynı key için daha önce işlemedik mi? (en yeni segment önce gelir, eski atlanır)
        if key != last_seen_key:
            result[key] = value  # İlk gelen = en yeni segment
            last_seen_key = key
        
        try:
            nk, nv = next(it)
            heapq.heappush(heap, (nk, seg_idx, nv, it))
        except StopIteration:
            pass
    
    return result

# Örnek: 3 segment (yeniden eskiye)
segment_newest = [("apple", 5), ("banana", 10), ("grape", 3)]
segment_mid    = [("apple", 3), ("cherry", 7), ("grape", 1)]
segment_oldest = [("apple", 1), ("banana", 8), ("date", 4)]

merged = merge_sstables(segment_newest, segment_mid, segment_oldest)
print("Birleştirilmiş sonuç (key -> en son değer):")
for k, v in sorted(merged.items()):
    print(f"  {k}: {v}")

Birleştirilmiş sonuç (key -> en son değer):
  apple: 5
  banana: 10
  cherry: 7
  date: 4
  grape: 3


### 3.3 Tombstone (Silinme İşareti)

Bir key'i silmek için, o key'e özel bir silme kaydı olan **tombstone** (mezar taşı) dosyaya eklenir.

- Segment'ler birleştirilirken tombstone, silinen key'in önceki tüm değerlerinin atılmasını tetikler.
- Tombstone en eski segment'e merge edildiğinde, tombstone'un kendisi de düşürülebilir.
- Bu mekanizma **LSM storage engine**'lerde gerçek silmenin nasıl çalıştığını açıklar: Veri hemen silinmez, yalnızca silineceği işaretlenir ve asıl silme işlemi compaction sırasında gerçekleşir.

> ⚠️ **GDPR/Veri Koruma Notu:** LSM engine'lerde silinen bir kayıt, tombstone tüm compaction seviyelerine yayılana kadar diskte kalmaya devam eder. Bu, "verinin gerçekten silindiğinden emin olunması" gereken yasal durumlar için sorun yaratabilir.

### 3.4 LSM-Tree (Log-Structured Merge-Tree)

Bu algoritmada oluşan cascade SSTable yapısı **LSM-tree** adıyla anılır:
- **1996'da Patrick O'Neil** tarafından yayımlandı.
- **RocksDB, Apache Cassandra, ScyllaDB, HBase** bu algoritmayı kullanır.
- Bunların tümü **Google Bigtable** makalesinden ilham almıştır (SSTable ve memtable terimlerini tanıtan makale).

**Temel Özellik:**
- Segment dosyaları **immutable** (değiştirilemez) olarak yazılır.
- Merge ve compaction **arka plan thread**'inde çalışır.
- Merge sürerken okumalar eski segment'lerden yanıtlanmaya devam eder.
- Merge tamamlandığında okumalar yeni birleştirilmiş segment'e yönlendirilir.

Segment dosyaları local disk yerine **object storage**'a (S3, GCS gibi) da yazılabilir. **SlateDB** ve **Delta Lake** bu yaklaşımı kullanır.

**Immutability ve Crash Recovery:**
- Çökme sırasında memtable yazılırken veya segment'ler birleştirilirken bir hata oluşursa, tamamlanmamış SSTable silinir ve işlem baştan başlatılır.
- Checksum'lar sayesinde bozulmuş/eksik log kayıtları tespit edilir ve atlanır.

---
## 4. Bloom Filter
### *Bloom Filtresi*

LSM storage'da, çok eski zamanda yazılmış veya hiç yazılmamış bir key'i okumak yavaş olabilir: storage engine tüm segment dosyalarını kontrol etmek zorunda kalır.

**Bloom filter**, her segment'te yer alan ve bir key'in o segment'te olup olmadığını **hızlı ama yaklaşık** biçimde kontrol eden bir veri yapısıdır.

### 4.1 Bloom Filter Nasıl Çalışır?

**Yapı:** Sabit boyutlu bir **bit dizisi** (bitmap).

**Ekleme (Build time):**
1. SSTable'daki her key için bir **hash fonksiyonu** uygulanır ve bir dizi sayı elde edilir.
2. Bu sayılar bit dizisinin **index**'leri olarak yorumlanır ve o bit'ler `1` yapılır.
3. Örnek: `handbag` key'i `(2, 9, 4)` hash değerleri üretsin → 2., 9. ve 4. bit'ler `1` olur.

**Sorgulama (Query time):**
1. Aranan key için aynı hash fonksiyonu uygulanır.
2. Çıkan index'lerdeki bit'ler kontrol edilir.
3. **En az bir bit `0` ise** → key kesinlikle bu SSTable'da **yoktur**. SSTable atlanır.
4. **Tüm bit'ler `1` ise** → key muhtemelen vardır (**ama kesin değil!**).

#### False Positive (Yanlış Pozitif)

Bir key aslında SSTable'da olmadığı halde, başka key'lerin set ettiği bit'ler üst üste geldiğinde "varmış gibi" görünebilir. Buna **false positive** denir.

- False positive durumunda sparse index'e bakılır ve blok okunur.
- Key yoksa biraz boş iş yapılmış olur, ama zarar yoktur.
- False positive **olasılığı** ayarlanabilir: Her key başına 10 bit ayırmak %1 false positive sağlar. Her ek 5 bit, ihtimali 10 kat düşürür.

> Bloom filter **false negative üretmez**: "Yok" diyorsa gerçekten yoktur. "Var" diyorsa belki vardır.

In [4]:
# Basit Bloom Filter implementasyonu
import hashlib

class BloomFilter:
    def __init__(self, size=100, num_hashes=3):
        self.size = size
        self.num_hashes = num_hashes
        self.bits = [0] * size
    
    def _get_hash_positions(self, key):
        positions = []
        for i in range(self.num_hashes):
            h = int(hashlib.md5(f"{key}_{i}".encode()).hexdigest(), 16)
            positions.append(h % self.size)
        return positions
    
    def add(self, key):
        for pos in self._get_hash_positions(key):
            self.bits[pos] = 1
    
    def might_contain(self, key):
        """False negative yok. False positive mümkün."""
        return all(self.bits[pos] == 1 for pos in self._get_hash_positions(key))

# SSTable içindeki key'leri Bloom filter'a ekle
bf = BloomFilter(size=200, num_hashes=3)
sstable_keys = ["handbag", "handoff", "handle", "handbook"]
for key in sstable_keys:
    bf.add(key)

# Sorgular
test_keys = ["handbag", "handiwork", "handsome", "handle", "xylophone"]
print("Key             | Bloom Filter Sonucu | Gerçekte Var mı?")
print("-" * 60)
for k in test_keys:
    bf_result = bf.might_contain(k)
    actual = k in sstable_keys
    status = "✅ Doğru" if bf_result == actual else "⚠️ False Positive!"
    print(f"{k:<16} | {'Var gibi' if bf_result else 'Yok':<20} | {'Var' if actual else 'Yok'}  {status}")

Key             | Bloom Filter Sonucu | Gerçekte Var mı?
------------------------------------------------------------
handbag          | Var gibi             | Var  ✅ Doğru
handiwork        | Yok                  | Yok  ✅ Doğru
handsome         | Yok                  | Yok  ✅ Doğru
handle           | Var gibi             | Var  ✅ Doğru
xylophone        | Yok                  | Yok  ✅ Doğru


---
## 5. Compaction Strategies (Sıkıştırma Stratejileri)

LSM storage engine'lerin önemli bir detayı: Compaction ne zaman yapılacak ve hangi SSTable'lar dahil edilecek?

### 5.1 Size-Tiered Compaction

- Daha yeni ve küçük SSTable'lar, daha eski ve büyük SSTable'larla birleştirilir.
- Örnek: Dört adet 256 MB SSTable → Bir adet ~898 MB SSTable (silmeler, overwrite'lar, TTL nedeniyle 1024 MB değil)
- Eski veriyi içeren SSTable'lar çok büyüyebilir ve birleştirilmesi çok fazla geçici disk alanı gerektirir.
- **Avantaj:** Yüksek yazma throughput'u destekler; çoğu veri yalnızca birkaç kez yeniden yazılır.
- **Kullanım yeri:** HBase (varsayılan), Cassandra (opsiyonel)

### 5.2 Leveled Compaction

- Büyük SSTable'lar yerine, SSTable boyutları sabit tutulur ve `L0, L1, L2, ...` gibi **level**'lara gruplandırılır.
- L0: En yeni yazılan veri.
- L1 ve sonrası: **Key range'e göre bölünmüş** SSTable'lar içerir. Örn: L1'de `a–m` ve `n–z` key'leri için iki ayrı SSTable.
- Her level için maksimum boyut sınırı vardır; sınır aşılınca level-i'den level-i+1'e merge yapılır.
- **Avantaj:** Daha az disk alanı kullanır, okuma için daha az SSTable kontrol edilmesi gerekir.
- **Kullanım yeri:** RocksDB, Cassandra (opsiyonel)

### Hangi Strateji Ne Zaman?

| Durum | Önerilen Strateji |
|-------|------------------|
| Çok fazla yazma, az okuma | Size-tiered |
| Okuma ağırlıklı iş yükü | Leveled |
| Az sayıda key sık, çok sayıda key nadir yazılıyorsa | Leveled |

Çoğu LSM implementasyonu farklı iş yükleri için farklı compaction stratejileri sunar.

### Kutucuk: Embedded Storage Engines

Çoğu veritabanı ağ üzerinden sorgu kabul eden bir servis olarak çalışır. Ancak **embedded database**'ler ağ API'si sunmaz. Bunlar, uygulama kodunuzla aynı process içinde çalışan **kütüphanelerdir** ve yerel disk dosyalarını okuyup yazar.

**Örnekler:** RocksDB, SQLite, LMDB, DuckDB, KùzuDB

**Kullanım Senaryoları:**
- **Mobile uygulamalar:** Kullanıcının yerel verisini saklamak için yaygın kullanılır.
- **Backend (sunucu tarafı):** Veri tek bir makineye sığıyorsa ve eşzamanlı transaction sayısı fazla değilse uygun olabilir.
- **Multitenant sistemler:** Her tenant küçük ve diğerlerinden tamamen ayrı ise her tenant için ayrı bir embedded veritabanı örneği kullanılabilir (Bluesky/AT Protocol bu yaklaşımı uygular).

---
## 6. B-Trees
### *B-Ağaçları*

B-tree, key-value kayıtlarını key'e göre okumak ve yazmak için en yaygın kullanılan veri yapısıdır.

- **1970'de** tanıtıldı, 10 yıl içinde "her yerde" olduğu söylendi.
- Hemen hemen tüm **ilişkisel veritabanlarında** (PostgreSQL, MySQL, Oracle, SQL Server) ve pek çok ilişkisel olmayan veritabanında standart index yapısıdır.
- SSTable gibi key'lere göre sıralıdır; hem key-value lookup hem de range query destekler.
- **Temel fark:** SSTable/LSM değiştirilemez segment dosyaları yazar, B-tree ise veriyi **yerinde (in-place) günceller**.

### 6.1 B-Tree Yapısı

B-tree, veritabanını sabit boyutlu **block** veya **page**'lere böler. Geleneksel olarak 4 KiB, PostgreSQL 8 KiB, MySQL ise 16 KiB kullanır.

**Temel Bileşenler:**
- **Root page:** B-tree'nin başlangıç noktası. Her key araması buradan başlar.
- **Internal pages:** Key aralıklarını ve child page referanslarını içerir.
- **Leaf pages:** Gerçek değerleri (veya değerlerin referanslarını) içerir.

**Branching Factor (Dallanma Faktörü):**
- Bir page'in kaç child page'e referans verdiğini gösterir.
- Genellikle **birkaç yüz** civarındadır.
- 500 branching factor ile 4 KiB page boyutunda 4 seviyeli B-tree **250 TB** veri depolayabilir!

**Key Arama:**
1. Root page'den başla.
2. Aranan key hangi range'e düşüyor? İlgili child page'e git.
3. Bu işlemi **leaf page**'e ulaşana kadar tekrarla.
4. Leaf page'den değeri oku.

**Key Güncelleme:**
- Key'i içeren leaf page bulunur.
- O page disk üzerinde yeni değeri içerecek şekilde **üzerine yazılır**.

**Key Ekleme:**
1. Yeni key'in düşeceği range'i içeren page bulunur.
2. Key o page'e eklenir.
3. Page doluysa **page split** (sayfa bölünmesi) yapılır:
   - Page iki yarı dolu page'e bölünür.
   - Parent page, yeni alt page'e referans içerecek şekilde güncellenir.
   - Bu bölünme root'a kadar yayılabilir; root bölündüğünde yeni bir root oluşturulur.

**B-tree her zaman dengelidir:** n key içeren B-tree'nin derinliği her zaman **O(log n)**'dir.

In [5]:
# B-tree'nin kavramsal gösterimi (Python'ın sortedcontainers ile basit simülasyon)
# Gerçek bir B-tree yerine, arama/ekleme mantığını göstermek amaçlı

class SimpleBTreeNode:
    def __init__(self, is_leaf=False):
        self.keys = []
        self.values = []
        self.children = []
        self.is_leaf = is_leaf

class SimpleBTree:
    """Eğitim amaçlı basit B-tree benzeri yapı (order=3)"""
    def __init__(self, order=3):
        self.root = SimpleBTreeNode(is_leaf=True)
        self.order = order  # Max keys per node
    
    def search(self, key, node=None):
        if node is None:
            node = self.root
        
        # Leaf'te mi arıyoruz?
        if node.is_leaf:
            for i, k in enumerate(node.keys):
                if k == key:
                    return node.values[i]
            return None
        
        # Internal node — hangi child'a gidilecek?
        for i, k in enumerate(node.keys):
            if key < k:
                return self.search(key, node.children[i])
        return self.search(key, node.children[-1])

# B-tree derinlik hesabı
import math

def btree_depth(n_keys, branching_factor):
    if n_keys <= 0:
        return 0
    return math.ceil(math.log(n_keys, branching_factor))

print("B-Tree Derinlik Hesabı (branching factor = 500, page = 4 KiB):")
print("-" * 55)
scenarios = [
    (1_000, "1 Bin kayıt"),
    (1_000_000, "1 Milyon kayıt"),
    (1_000_000_000, "1 Milyar kayıt"),
    (250_000_000_000, "250 TB (4KiB pages, BF=500)"),
]
for n, label in scenarios:
    depth = btree_depth(n, 500)
    print(f"  {label:<35} → Derinlik: {depth}")

print("\nSonuç: Trilyonlarca kayıtta bile yalnızca 3-4 page okumak yeterli!")

B-Tree Derinlik Hesabı (branching factor = 500, page = 4 KiB):
-------------------------------------------------------
  1 Bin kayıt                         → Derinlik: 2
  1 Milyon kayıt                      → Derinlik: 3
  1 Milyar kayıt                      → Derinlik: 4
  250 TB (4KiB pages, BF=500)         → Derinlik: 5

Sonuç: Trilyonlarca kayıtta bile yalnızca 3-4 page okumak yeterli!


### 6.2 B-Tree'yi Güvenilir Kılmak — Write-Ahead Log (WAL)

B-tree'nin temel yazma işlemi: Bir page'i yeni veriyle **disk üzerinde üzerine yaz**.

**Tehlike:** Page split sırasında birden fazla page yazılması gerekir. Eğer yalnızca bazı page'ler yazılmış durumdayken sistem çökerse:
- **Orphan page:** Hiçbir parent'ın child'ı olmayan bir page oluşabilir.
- **Torn page:** Donanım atomik olarak tam bir page yazamazsa, yarı yazılmış page oluşabilir.

**Çözüm: Write-Ahead Log (WAL)**
- WAL, disk üzerindeki **append-only bir dosyadır**.
- Her B-tree değişikliği, ağaca uygulanmadan önce **önce WAL'a yazılır**.
- Veritabanı çökmeden sonra yeniden başladığında, B-tree'yi tutarlı bir duruma geri getirmek için WAL kullanılır.
- Dosya sistemlerinde benzer mekanizmaya **journaling** denir.

**Durabilty (Kalıcılık):**
- B-tree implementasyonları genellikle değiştirilmiş page'leri hemen diske yazmaz; bir süre **bellekte buffer**'lar.
- WAL, veri diske `fsync` ile flush edildiği sürece çökmede kayıp yaşanmayacağını garantiler.

### 6.3 B-Tree Varyantları

B-tree'ler onlarca yıldır kullanıldığından, zaman içinde pek çok varyant geliştirilmiştir:

**1. Copy-on-Write (COW) Scheme:**
- WAL tutmak yerine bazı veritabanları (örn: **LMDB**) copy-on-write kullanır.
- Değiştirilen page **aynı konuma yazılmaz**, farklı bir konuma yazılır.
- Parent page'ler de yeni konumu gösterecek şekilde güncellenir.
- Hem crash recovery hem de **concurrency control** için yararlıdır.

**2. Abbreviated Keys (Kısaltılmış Key'ler):**
- Internal node'larda tam key yerine **kısaltılmış key** saklanabilir.
- Key'ler yalnızca range boundary olarak işlev görür; tam değerleri gerekmez.
- Daha fazla key bir page'e sığar → branching factor artar → tree daha sığ olur.

**3. Sıralı Leaf Page Yerleşimi:**
- Bazı B-tree implementasyonları, leaf page'leri **disk üzerinde sıralı biçimde** yerleştirmeye çalışır.
- Key range üzerinde sıralı tarama (scan) hızlanır; daha az disk seek gerekir.
- Ancak tree büyüdükçe bu sırayı korumak zorlaşır.

**4. Sibling Pointer'lar:**
- Her leaf page, sol ve sağ kardeş page'lerine referans içerebilir.
- Parent page'lere atlamadan key'ler sırayla taranabilir.

---
## 7. B-Tree vs LSM-Tree Karşılaştırması
### *B-Ağacı ile LSM-Ağacı Karşılaştırması*

Genel kural: **LSM-tree yazma ağırlıklı** uygulamalar için, **B-tree okuma açısından daha hızlı**.

Ancak benchmark'lar iş yükünün detaylarına çok duyarlıdır. Ayrıca bu iki yaklaşım arasında kesin bir "ya biri ya diğeri" seçimi yoktur; bazı storage engine'ler her iki yaklaşımın özelliklerini karıştırır.

### 7.1 Read Performance (Okuma Performansı)

**B-tree'de okuma:**
- Her level'da bir page okunur → Tahmin edilebilir ve genellikle hızlıdır.
- Range query'ler basit ve hızlıdır; tree'nin sıralı yapısından yararlanır.

**LSM'de okuma:**
- Compaction'ın farklı aşamalarındaki birden fazla SSTable kontrol edilebilir.
- Bloom filter'lar gereksiz disk I/O'sunu azaltır.
- Range query'ler tüm segment'leri paralel taramak ve sonuçları birleştirmek gerektirir.
- Bloom filter range query'lerde işe yaramaz (her olası key için hash hesaplamak pratik değil).

**Backpressure (Geri Baskı):**
- LSM engine'lerde memtable dolduğunda (compaction yetişemediğinde) tüm okuma ve yazmalar **askıya alınabilir**.
- RocksDB bu durumda backpressure uygular.

**Modern SSD ve NVMe:**
- NVMe SSD'ler PCIe üzerinden SATA'ya göre çok daha hızlıdır.
- Hem LSM hem B-tree yüksek read throughput sağlayabilir, ama parallelism'den yararlanmak için dikkatli tasarım gerekir.

### 7.2 Sequential vs Random Writes (Sıralı ve Rastgele Yazma)

**B-tree yazma:** Uygulamanın key'leri tüm key space'e dağılmış yazması → Disk operasyonları da **rastgele dağılır**. Yazılması gereken page'ler diskin herhangi bir yerinde olabilir → **Random writes**.

**LSM yazma:** Tüm segment dosyaları bir seferde yazılır (memtable flush veya compaction). Bunlar B-tree page'inden çok daha büyük → **Sequential writes**.

**Diskler sıralı yazmayı neden daha hızlı yapar?**
- **HDD (Spinning disk):** Random write için disk kafası mekanik olarak hareket etmek zorunda — birkaç milisaniye gecikme. Sequential write çok daha hızlı.
- **SSD / NVMe:** Mekanik kısıtlama yok, ama SSD'ler de sequential write'ta daha hızlı:
  - Flash memory **blok** bazında silinebilir (genellikle 512 KiB).
  - Sequential workload büyük parçalar yazar → Silme basit. Random workload geçerli/geçersiz page karışımı oluşturur → SSD **Garbage Collection (GC)** daha çok çalışır → Bant genişliği GC'ye harcanır, SSD daha hızlı eskir.

### 7.3 Write Amplification (Yazma Büyütmesi)

**Write Amplification:** Uygulamadan gelen tek bir yazma isteğinin, altındaki diskte birden fazla I/O operasyonuna dönüşmesi.

**LSM-tree'de write amplification:**
- Bir değer şu sırayla yazılır:
  1. Dayanıklılık için **WAL**'a.
  2. **Memtable flush** edildiğinde SSTable'a.
  3. Her **compaction**'da yeniden.
- Not: Value'lar key'lerden çok büyükse, value'ları key'lerden ayrı saklamak (key-value separation) overhead'i azaltabilir.

**B-tree'de write amplification:**
- Her veri en az **iki kez** yazılır: Bir kez WAL'a, bir kez ağaç page'ine.
- Page'de yalnızca birkaç byte değişse bile bazen **tüm page** yeniden yazılır.

**Write Amplification Formülü:**
```
Write Amplification = (Diske toplam yazılan byte) / (Append-only log olarak yazılırdı byte)
```

**Neden Önemli?**
- Yazma ağırlıklı uygulamalarda bottleneck, diskin **yazma bant genişliği** olabilir.
- Write amplification ne kadar yüksekse, mevcut disk bant genişliğiyle saniyede o kadar az yazma yapılır.
- SSD'ler için: Daha düşük write amplification = SSD daha yavaş eskir.

**LSM vs B-tree:** Tipik iş yüklerinde LSM daha düşük write amplification'a sahiptir (tam page yazmak zorunda değil, SSTable bloklarını sıkıştırabilir).

### 7.4 Disk Space Usage (Disk Alanı Kullanımı)

**B-tree fragmentation (Parçalanma):**
- Çok sayıda key silinirse, dosya artık B-tree tarafından kullanılmayan **page**'ler içerebilir.
- Yeni eklemeler bu boş page'leri kullanabilir, ama page'ler dosyanın ortasında olduğu için OS'e iade edilemez → Dosyada boş alan birikir.
- PostgreSQL bu sorunu çözmek için **vacuum** arka plan süreci çalıştırır.

**LSM daha iyi:**
- Compaction zaten veri dosyalarını yeniden yazar → Parçalanma daha az sorun.
- SSTable block'ları daha iyi sıkıştırılabilir → Genellikle B-tree'ye göre daha küçük disk dosyaları.
- **Leveled compaction** disk alanını verimli kullanır.
- **Size-tiered compaction** compaction sırasında geçici olarak çok fazla disk alanı kullanır.

**Silme ve Snapshot:**
- Tombstone tüm compaction seviyelerine yayılana kadar silinen kayıtlar diskte kalmaya devam eder.
- Öte yandan, immutable SSTable segment dosyaları **snapshot almayı** kolaylaştırır: Belirli bir andaki segment listesini kaydet, bu dosyaları silme. B-tree'de bu çok daha karmaşıktır.

In [6]:
# B-Tree vs LSM-Tree Karşılaştırma Özeti
import pandas as pd

comparison = {
    "Özellik": [
        "Read throughput",
        "Write throughput",
        "Write amplification",
        "Range queries",
        "Disk space (fragmentation)",
        "Compaction overhead",
        "Crash recovery",
        "Snapshot / backup",
        "SSD wear",
    ],
    "B-Tree": [
        "✅ Daha hızlı (tahmin edilebilir)",
        "🔸 Orta",
        "🔴 Yüksek (WAL + tüm page)",
        "✅ Basit ve hızlı",
        "🔴 Fragmentation riski",
        "✅ Yok",
        "🔸 WAL ile",
        "🔴 Zor (in-place update)",
        "🔴 Daha fazla tüketim",
    ],
    "LSM-Tree": [
        "🔸 Bloom filter ile iyi, range'de daha zayıf",
        "✅ Yüksek (sequential write)",
        "✅ Daha düşük",
        "🔸 Mümkün ama paralel segment tarama gerekir",
        "✅ Compaction düzenli temizler",
        "🔴 Backpressure riski",
        "✅ Immutable segment + WAL",
        "✅ Kolay (immutable dosyalar)",
        "✅ Daha az tüketim",
    ],
}

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

                   Özellik                           B-Tree                                    LSM-Tree
           Read throughput ✅ Daha hızlı (tahmin edilebilir) 🔸 Bloom filter ile iyi, range'de daha zayıf
          Write throughput                           🔸 Orta                 ✅ Yüksek (sequential write)
       Write amplification        🔴 Yüksek (WAL + tüm page)                                ✅ Daha düşük
             Range queries                 ✅ Basit ve hızlı 🔸 Mümkün ama paralel segment tarama gerekir
Disk space (fragmentation)            🔴 Fragmentation riski               ✅ Compaction düzenli temizler
       Compaction overhead                            ✅ Yok                        🔴 Backpressure riski
            Crash recovery                        🔸 WAL ile                   ✅ Immutable segment + WAL
         Snapshot / backup          🔴 Zor (in-place update)                ✅ Kolay (immutable dosyalar)
                  SSD wear             🔴 Daha fazla tüketim     

---
## 8. Multicolumn and Secondary Indexes
### *Çok Sütunlu ve İkincil İndeksler*

Şimdiye kadar yalnızca **primary-key index**'leri (bir kaydı tekil olarak tanımlayan key'e göre index) ele aldık.

**Secondary index (İkincil index):**
- Aynı tablo üzerinde SQL `CREATE INDEX` komutuyla birden fazla secondary index oluşturulabilir.
- Primary key dışındaki sütunlara göre arama yapılabilir.
- Örnek: `user_id` kolonuna secondary index — bir kullanıcıya ait tüm satırları bulmak için.

**Fark:**
- Primary key index: Index değerleri tekil (unique).
- Secondary index: Index değerleri tekil **olmak zorunda değil** — aynı index girişi altında birden fazla kayıt olabilir.

Bu iki şekilde çözülebilir:
1. Index'teki her değer, eşleşen kayıt ID'lerinin listesidir (full-text index'teki **postings list** gibi).
2. Her entry'ye kayıt ID'si eklenerek tekil yapılır.

Hem B-tree hem de LSM tabanlı storage, secondary index implementasyonu için kullanılabilir.

---
## 9. Storing Values Within the Index
### *Index İçinde Değer Saklamak*

Index'in key'i, sorguların arama yaptığı şeydir. Key'e ek olarak index içinde ne saklandığına göre üç tür vardır:

### 9.1 Clustered Index

- Gerçek veri (satır, belge, vertex) **doğrudan index yapısının içinde** saklanır.
- Örnek: MySQL'in **InnoDB** storage engine'inde bir tablonun primary key'i her zaman clustered index'tir. **SQL Server**'da da tablo başına bir clustered index tanımlanabilir.

### 9.2 Heap File Yaklaşımı

- Index'in değeri gerçek veriye işaret eden bir **referanstır**: Ya primary key, ya da disk üzerindeki fiziksel konumu.
- Satırların saklandığı yer **heap file** olarak adlandırılır. Heap file sırasız veri saklar.
- **PostgreSQL** heap file yaklaşımını kullanır.
- **MySQL InnoDB:** Secondary index'ler, primary key'i referans olarak saklar → Sorgu önce secondary index'ten primary key'i bulur, sonra clustered primary key index'inden gerçek satırı alır.

### 9.3 Covering Index (Kapsayan Index)

- İki yaklaşım arasında orta yol: **Bazı sütunları** index içinde sakla, tam satırı heap'te tut.
- Bu sayede bazı sorgular **yalnızca index'ten yanıtlanabilir** (heap veya primary key'e bakılmaz) → Sorgu daha hızlı.
- Index'in sorguyu kapsadığı söylenir: **"index covers the query"**.
- Dezavantaj: Veri tekrarlanması → Daha fazla disk alanı ve yazma yavaşlar.

---
## 10. Keeping Everything in Memory
### *Her Şeyi Bellekte Tutmak — In-Memory Databases*

Bu bölüme kadar ele alınan tüm veri yapıları disk sınırlamalarının ürünüdür. Disk ile RAM arasındaki fark:
- **Disk:** Dayanıklı, gigabyte başına düşük maliyet, ama yavaş ve dikkatli yerleşim gerektirir.
- **RAM:** Hızlı, ama güç kesilirse veriler kaybolur ve daha pahalıdır.

RAM fiyatları düştükçe, birçok veri kümesi tek başına bellekte saklanabilir hale geldi → **In-memory databases**.

### Dayanıklılık Stratejileri

**Veri kaybını önleme yöntemleri:**
- Pil destekli RAM (battery-powered RAM) — özel donanım.
- Değişiklikleri diske **log** olarak yaz.
- Periyodik **snapshot** al.
- In-memory durumu diğer makinelere **replicate** et.

Yeniden başlatıldığında veri diskten veya bir replica'dan yüklenir. Disk yalnızca **dayanıklılık için** kullanılır; okumalar tamamen bellekten yapılır.

### Neden Daha Hızlı?

Sezgiye aykırı bir gerçek: In-memory database'lerin performans avantajı **diskten okumama** nedenli **değildir**. Çünkü disk tabanlı bir veritabanı da yeterli RAM varsa diskteki blokları önbellekte tutabilir.

Gerçek avantaj: **In-memory veri yapılarını diske yazılabilir bir formata encoding etme overhead'ini ortadan kaldırır**.

### Örnekler

| Veritabanı | Özellik |
|------------|----------|
| **Memcached** | Yalnızca caching, veri kaybı kabul edilebilir |
| **Redis** | Disk'e async yazarak zayıf dayanıklılık |
| **Couchbase** | Disk'e async yazarak zayıf dayanıklılık |
| **VoltDB** | Relational model, tam dayanıklılık |
| **SingleStore** | Relational model, tam dayanıklılık |
| **Oracle TimesTen** | Relational in-memory database |
| **RAMCloud** | Log-structured in-memory key-value store |

**Farklı veri modelleri:** Redis, priority queue ve set gibi veri yapıları için veritabanı benzeri arayüz sunar. Bunlar disk tabanlı index'lerle implement etmesi zor yapılardır. Tüm veriyi bellekte tutmak, implementasyonu basitleştirir.

---
## 11. Data Storage for Analytics
### *Analitik için Veri Depolama*

Data warehouse'un veri modeli genellikle **ilişkisel**dir çünkü SQL analitik sorgular için iyi bir tercih. Yüzeysel olarak, data warehouse ile ilişkisel OLTP veritabanı aynı görünür (her ikisi de SQL arayüzü). Ama içleri çok farklıdır; farklı sorgu kalıpları için optimize edilmişlerdir.

Birçok veritabanı satıcısı ya transaction processing ya da analytics'i desteklemeye odaklanır, ikisini birden değil.

**HTAP (Hybrid Transactional/Analytical Processing):**
- Microsoft SQL Server, SAP HANA ve SingleStore gibi bazı veritabanları her ikisini de tek üründe sunar.
- Ancak bu HTAP sistemleri giderek ortak bir SQL arayüzü üzerinden erişilen **iki ayrı storage/query engine**'e dönüşmektedir.

### 11.1 Cloud Data Warehouses
### *Bulut Veri Ambarları*

**Geleneksel on-premises:** Teradata, Vertica, SAP HANA → Ticari lisanslarla.

**Cloud-native (yalnızca bulut):** Google BigQuery, Amazon Redshift, Snowflake → Ölçeklenebilir bulut altyapısından (object storage, serverless hesaplama) yararlanır.

Modern cloud data warehouse mimarisi genellikle şu katmanlara ayrılır:

**Query engine:**
- SQL sorgularını parse eder, optimize eder ve yürütme planlarına dönüştürür.
- Paralel, dağıtık veri işleme görevleri gerektirir.
- Örnekler: Trino, Apache DataFusion, Presto
- Bazı query engine'ler Spark veya Flink gibi execution framework'leri kullanır.

**Storage format:**
- Bir tablonun satırlarının byte olarak nasıl encode edildiğini belirler.
- Genellikle **object storage** veya dağıtık dosya sistemine kaydedilir.
- Örnekler: **Parquet, ORC, Lance, Nimble**

**Table format:**
- Parquet gibi storage format dosyaları genellikle bir kez yazılıp **immutable** kalır.
- Satır ekleme/silmeyi desteklemek için table format kullanılır.
- Örnekler: **Apache Iceberg**, Databricks Delta Lake
- Gelişmiş özellikler: **Time travel** (geçmişteki tablonun sorgulanması), GC, transaction.

**Data catalog:**
- Hangi dosyaların bir tabloyu oluşturduğunu table format belirlerken, **catalog** bir veritabanında hangi tabloların bulunduğunu tanımlar.
- Tablo oluşturma, yeniden adlandırma ve silme işlemleri için kullanılır.
- Örnekler: Snowflake Polaris, Databricks Unity Catalog, Apache Iceberg Catalog
- Genellikle REST arayüzü üzerinden sorgulanabilir bağımsız servisler.

---
## 12. Column-Oriented Storage
### *Sütun Odaklı Depolama*

Data warehouse'larda fact table'lar trilyonlarca satır ve petabytes veri içerebilir. Analitik bir sorgu genellikle 100+ sütunlu tablonun yalnızca 3-5 sütununa erişir.

**Örnek sorgu:**
```sql
SELECT dim_date.weekday, dim_product.category, SUM(fact_sales.quantity)
FROM fact_sales
  JOIN dim_date ON fact_sales.date_key = dim_date.date_key
  JOIN dim_product ON fact_sales.product_sk = dim_product.product_sk
WHERE dim_date.year = 2024
  AND dim_product.category IN ('Fresh fruit', 'Candy')
GROUP BY dim_date.weekday, dim_product.category;
```
Bu sorgu `fact_sales`'in yalnızca 3 sütununa erişiyor: `date_key`, `product_sk`, `quantity`.

### Row-Oriented vs Column-Oriented

**Row-oriented (satır odaklı) — OLTP'de yaygın:**
- Bir satırdaki tüm değerler birbirinin yanında saklanır.
- Tüm satırı okumak verimli, ama sadece birkaç sütun lazımsa diğerlerini de okumak zorundasın.

**Column-oriented (sütun odaklı) — OLAP/DW'de yaygın:**
- Her sütunun tüm değerleri birbirinin yanında saklanır.
- Sütun-odaklı storage'da, sorgunun ihtiyaç duyduğu sütunlar parse edilir — diğerleri **hiç okunmaz**.

**Önemli Kural:** Sütun storage'da tüm sütunlar aynı sırada satırları içermelidir. Böylece 23. satırı yeniden oluşturmak için her sütundaki 23. elemandan alınır.

In [7]:
# Row-oriented vs Column-oriented erişim farkı

# --- Row-oriented (satır bazlı) ---
rows_data = [
    {"date_key": 20240101, "product_sk": 31, "store_sk": 5, "quantity": 3, "net_price": 12.50},
    {"date_key": 20240101, "product_sk": 68, "store_sk": 2, "quantity": 1, "net_price": 5.00},
    {"date_key": 20240102, "product_sk": 31, "store_sk": 5, "quantity": 5, "net_price": 20.83},
    {"date_key": 20240103, "product_sk": 69, "store_sk": 3, "quantity": 2, "net_price": 8.00},
]

# Row-oriented: Sadece quantity lazım, ama tüm satır okunuyor
print("Row-oriented — tüm sütunlar okunuyor:")
total_row = 0
for row in rows_data:
    print(f"  Okunan: {list(row.keys())} → Kullanılan: quantity={row['quantity']}")
    total_row += row['quantity']
print(f"  Toplam quantity: {total_row}")

print()

# --- Column-oriented (sütun bazlı) ---
col_date_key   = [20240101, 20240101, 20240102, 20240103]  # 1 sütun
col_product_sk = [31, 68, 31, 69]                          # 1 sütun
col_store_sk   = [5, 2, 5, 3]                              # 1 sütun  
col_quantity   = [3, 1, 5, 2]                              # 1 sütun
col_net_price  = [12.50, 5.00, 20.83, 8.00]               # 1 sütun

# Column-oriented: Sadece quantity sütunu okunuyor!
print("Column-oriented — sadece quantity sütunu okunuyor:")
total_col = sum(col_quantity)  # Diğer sütunlara dokunmadık!
print(f"  Okunan: col_quantity = {col_quantity}")
print(f"  Toplam quantity: {total_col}")
print()
print("I/O tasarrufu: Row-oriented 5 sütun okurken, column-oriented 1 sütun okudu!")

Row-oriented — tüm sütunlar okunuyor:
  Okunan: ['date_key', 'product_sk', 'store_sk', 'quantity', 'net_price'] → Kullanılan: quantity=3
  Okunan: ['date_key', 'product_sk', 'store_sk', 'quantity', 'net_price'] → Kullanılan: quantity=1
  Okunan: ['date_key', 'product_sk', 'store_sk', 'quantity', 'net_price'] → Kullanılan: quantity=5
  Okunan: ['date_key', 'product_sk', 'store_sk', 'quantity', 'net_price'] → Kullanılan: quantity=2
  Toplam quantity: 11

Column-oriented — sadece quantity sütunu okunuyor:
  Okunan: col_quantity = [3, 1, 5, 2]
  Toplam quantity: 11

I/O tasarrufu: Row-oriented 5 sütun okurken, column-oriented 1 sütun okudu!


### 12.1 Column Compression
### *Sütun Sıkıştırma*

Sütun odaklı storage sıkıştırmaya çok uygun bir yapıya sahiptir. Bir sütundaki değerlerde çok sayıda tekrar vardır (örn: milyarlarca satış transaction'ı ama yalnızca 100.000 farklı ürün).

#### Bitmap Encoding (Bit Eşlem Kodlama)

Data warehouse'larda özellikle etkili bir sıkıştırma tekniği:

1. **n adet farklı değer içeren sütun** → **n adet bitmap**'e dönüştürülür.
2. Her bitmap, bir değer için tüm satırlar boyunca 1/0 içerir.
3. Bir satırda o değer varsa → 1, yoksa → 0.

#### Run-Length Encoding (Ardışık Tekrar Kodlama)

Bitmap'ler genellikle **sparse** (çok 0 içeren) yapıdadır. Run-length encoding ile ardışık 0'lar veya 1'lerin sayısı tutulur.

#### Roaring Bitmaps

İki bitmap gösterimi arasında geçiş yapar; hangisi daha kompakt ise onu kullanır.

#### Bitmap Index'lerin Sorgu Avantajı

```sql
WHERE product_sk IN (31, 68, 69)
```
→ `product_sk=31`, `product_sk=68`, `product_sk=69` için 3 bitmap yükle, bitwise **OR** al.

```sql
WHERE product_sk = 30 AND store_sk = 3
```
→ `product_sk=30` ve `store_sk=3` bitmap'lerini yükle, bitwise **AND** al.

Bu işlemler tüm CPU'ların desteklediği bitwise operasyonlarla **son derece hızlı** yapılır.

> ⚠️ **Karıştırma:** Column-oriented storage ile **wide-column** (column-family) veri modelini karıştırmayın! Wide-column (Google Bigtable, HBase, Accumulo) bir satırın binlerce kolona sahip olabildiği bir modeldir ve **row-oriented** olarak saklanır.

### 12.2 Sort Order in Column Storage
### *Sütun Depolamada Sıralama*

Column store'da satırların saklanma sırası başlangıçta önemli değildir (insertion order kullanılabilir). Ama bir sıra dayatmak ek avantajlar sağlar:

**Önemli Kural:** Sütunları bağımsız olarak sıralamak anlamsız. Sıralama **tüm satır bazında** yapılmalı (her sütun aynı satır sırasını korumalı).

**Sıralama Avantajları:**
1. **Sorgu hızı:** Sorgular çoğunlukla belirli tarih aralığını hedefler → `date_key` birincil sort key yapılırsa yalnızca o aralığın blokları okunur.
2. **İkincil sort key:** `date_key` aynı ise `product_sk` sıralama belirler. Aynı tarih + ürün kombinasyonları storage'da bir arada tutulur.
3. **Compression:** Birincil sort kolonunda ardışık tekrarlar çok olur → Run-length encoding son derece etkili olur. Milyarlarca satır bile birkaç kilobayta sıkışabilir.
4. **İkincil/üçüncül sort kolonlar** daha az tekrara sahip olur, compression avantajı azalır. Ama ilk birkaç kolonun sıralı olması gene de genel bir kazanç.

### 12.3 Writing to Column-Oriented Storage
### *Sütun Depolamaya Yazma*

Data warehouse'larda okumalar genellikle büyük agregasyon sorgularıdır. Column-oriented storage, compression ve sorting bunları hızlandırır.

**Yazma zorlukları:** Data warehouse'larda yazma genellikle **toplu import (bulk ETL)** şeklindedir. Sıralı tablonun ortasına tek bir satır yazmak çok verimsiz olurdu — tüm sıkıştırılmış sütunları ekleme noktasından itibaren yeniden yazmak gerekirdi.

**Çözüm:** Log-structured yaklaşım kullanılır:
1. Tüm yazmalar önce **row-oriented, sıralı, in-memory store**'a gider.
2. Yeterli yazma biriktiğinde, diskteki column-encoded dosyalarla **merge** edilir ve toplu olarak yeni dosyalar yazılır.
3. Eski dosyalar immutable olduğu için **object storage** bu dosyaları saklamak için çok uygun.

Sorgu engine, disk üzerindeki column datası ile bellekteki son yazmalar arasındaki farkı kullanıcıdan gizler. Insert, update veya delete yapılan veri sonraki sorgularda hemen yansır. **Snowflake, Vertica, Apache Pinot, Apache Druid** bu yaklaşımı kullanır.

---
## 13. Query Execution: Compilation and Vectorization
### *Sorgu Yürütme: Derleme ve Vektörleştirme*

Analitik için karmaşık bir SQL sorgusu, birden fazla aşamadan (**operator**) oluşan bir **query plan**'a dönüştürülür. Bu operatörler paralel yürütme için birden fazla makineye dağıtılabilir.

Milyonlarca satırı taraması gereken analitik sorgularda **CPU zamanı** da bir darboğaz olabilir. İki etkili yaklaşım geliştirilmiştir:

### 13.1 Query Compilation (Sorgu Derleme)

- Query engine, SQL sorgusunu alır ve onu **çalıştırmak için kod üretir**.
- Üretilen kod satır satır iterate eder, ilgili kolonlardaki değerlere bakar, karşılaştırma/hesaplama yapar ve koşulları sağlayanları çıktı buffer'ına kopyalar.
- Query engine üretilen kodu **makine koduna derler** (genellikle LLVM gibi mevcut bir derleyici kullanılarak).
- Bu yaklaşım, Java Virtual Machine'deki **JIT (Just-in-Time) compilation**'a benzer.

### 13.2 Vectorized Processing (Vektörleştirilmiş İşlem)

- Sorgu yorumlanır (derlenmez), ama **satır satır değil, sütundan bir batch değer** işlenerek hız kazanılır.
- Önceden tanımlanmış operatörler set halinde argümanlar alır ve toplu sonuçlar döndürür.
- Örnek: `product_sk` sütunu ve bir ürün ID'si eşitlik operatörüne geçirilir → Eşleşen satırlar için 1, diğerleri için 0 içeren **bitmap** döner.
- İki bitmap bitwise AND ile birleştirilerek sonuç elde edilir.

### CPU Optimizasyon Teknikleri

Her iki yaklaşım da modern CPU özelliklerinden yararlanır:
- **Sequential memory access:** Cache miss'leri azaltmak için rastgele erişim yerine sıralı bellek erişimi.
- **Pipelined execution:** CPU instruction pipeline'ını meşgul tutmak, branch misprediction'ı önlemek için tight inner loop'lar.
- **SIMD (Single-Instruction, Multiple Data):** Tek bir CPU komutuyla birden fazla veri üzerinde işlem yapılır.
- **Sıkıştırılmış veriyle doğrudan çalışma:** Veriyi ayrı bir in-memory representation'a decode etmeye gerek kalmaz.

---
## 14. Materialized Views and Data Cubes
### *Materyalize Edilmiş Görünümler ve Veri Küpleri*

### 14.1 Materialized Views

**Virtual view:** Sorgu yazmanın kısayolu. Bir view okunduğunda SQL engine, view'ün altındaki sorguya genişleterek işler.

**Materialized view:** Sorgu sonuçlarının gerçek bir kopyası, **diske yazılır**.

- Altta yatan veri değiştiğinde materialized view de güncellenir.
- Bazı veritabanları bunu otomatik yapar.
- **Materialize** gibi özel sistemler materialized view bakımını otomatize eder.
- Yazma işlemi artar, ama aynı sorgu tekrar tekrar çalışacaksa okuma hızlanır.

### 14.2 Data Cube (OLAP Cube)

**Materialized aggregate (materyalize edilmiş toplam):** Veri ambarlarında sık kullanılan materialized view türü.

Sorgular çoğunlukla `COUNT, SUM, AVG, MIN, MAX` gibi agregasyon fonksiyonları içerir. Aynı agregatlar çok sayıda sorgu tarafından kullanılıyorsa, ham veriyi her seferinde işlemek israftır.

**Data Cube (OLAP Cube):** Farklı boyutlara göre gruplandırılmış agregatların ızgara yapısında saklanması.

Örnek: `date_key` ve `product_sk` için 2 boyutlu data cube:
- Bir eksende tarih, diğerinde ürün.
- Her hücre: O tarih-ürün kombinasyonu için agregatı (örn: `SUM(net_price)`) içerir.
- Her satır veya sütun boyunca toplam alınarak bir boyut elenir.

**5 boyutlu data cube:** date × product × store × promotion × customer → Her kombinasyon için satış rakamları.

**Avantaj:** Belirli sorgular son derece hızlanır — önceden hesaplanmış!

**Dezavantaj:** Ham veri kadar esnek değil. "Fiyatı 100$'dan yüksek ürünlerin satış oranı" gibi bir sorgu, fiyat bir boyut değilse cevaplandırılamaz.

Bu nedenle çoğu data warehouse mümkün olduğunca ham veriyi korur ve data cube'ları yalnızca belirli sorgular için performans artışı olarak kullanır.

---
## 15. Multidimensional and Full-Text Indexes
### *Çok Boyutlu ve Tam Metin İndeksleri*

B-tree ve LSM-tree tek bir attribute üzerinde range query'ye izin verir. Ama bazen tek attribute yetmez.

### 15.1 Concatenated Index (Birleştirilmiş İndeks)

Çok sütunlu index'lerin en yaygın türü: Birden fazla field, **tek bir key**'e birleştirilir (eski telefon rehberi örneği: `(soyad, isim) → telefon numarası`).

- `(lastname, firstname)` index'i: Belirli bir soyadına sahip kişileri veya belirli bir soyad+isim kombinasyonunu bulmak için kullanılabilir.
- Ama yalnızca belirli bir **isim**e sahip kişileri bulmak için **işe yaramaz** — index önce soyadına göre sıralı.

### 15.2 Multidimensional Indexes (Çok Boyutlu İndeksler)

Birden fazla kolonu aynı anda sorgulamaya izin verir. **Coğrafi veri** için özellikle önemli.

Örnek: Bir harita uygulamasında kullanıcının görüntülediği dikdörtgen alandaki tüm restoranları bul:
```sql
SELECT * FROM restaurants
WHERE latitude  BETWEEN 51.4946 AND 51.5079
  AND longitude BETWEEN -0.1162 AND -0.1004;
```

`latitude` ve `longitude` üzerindeki concatenated index bu sorgu için verimsiz. Ya yalnızca latitude'a (herhangi bir longitude'da), ya da yalnızca longitude'a göre sonuç verebilir — ikisini aynı anda **yapamaz**.

**Çözümler:**

1. **Space-filling curve (Uzayı dolduran eğri):** 2 boyutlu konumu tek bir sayıya çevir, sonra B-tree index kullan.

2. **R-tree (Rectangle tree):** Uzayı bölerek yakın veri noktalarını aynı subtree'de gruplandırır. PostGIS, PostgreSQL'in GiST (Generalized Search Tree) indexleme altyapısını kullanarak R-tree uygular.

3. **Bkd-tree:** R-tree'nin varyantı.

4. **Hexagonal grid:** Uber'in H3 sistemi gibi düzenli üçgen/kare/altıgen ızgaraları.

**Çok boyutlu index diğer kullanımlar:**
- E-ticaret: `(red, green, blue)` boyutlarında 3D index → Belirli renk aralığındaki ürünler.
- Hava durumu: `(date, temperature)` üzerinde 2D index → Verilen yılda 25-30°C arası tüm ölçümler.

---
## 16. Full-Text Search
### *Tam Metin Arama*

Full-text search, bir metin belgesi koleksiyonunda metnin herhangi bir yerinde geçebilen anahtar kelimeleri aramayı sağlar.

**Zorluklara Genel Bakış:**
- Dil-spesifik işleme: Bazı Asya dilleri boşluk/noktalama kullanmaz; kelimeler için model gerekir.
- Yazım hataları ve eş anlamlılar.
- Farklı dilbilgisel formlar.

Full-text search, kavramsal olarak **çok boyutlu bir sorgu** olarak düşünülebilir: Metinde geçebilecek her kelime (**term**) bir boyuttur. Bir belge term X içeriyorsa, X boyutunda değeri 1; içermiyorsa 0.

### 16.1 Inverted Index (Ters İndeks)

Arama motorlarının bu tür sorguları yanıtlamak için kullandığı veri yapısı: **inverted index**.

- **Key:** Bir term (kelime).
- **Value:** O terimi içeren tüm belgelerin ID listesi → **postings list**.
- Belge ID'leri sıralı ise postings list sparse bitmap olarak da gösterilebilir.

**"Red apples" içeren belgeler:**
1. `red` term'ü için postings list (bitmap) yükle.
2. `apples` term'ü için postings list (bitmap) yükle.
3. Bitwise AND hesapla → Her iki terimi de içeren belgeler.

**Lucene (Elasticsearch ve Solr'ın kullandığı full-text index engine):**
- Term → postings list eşlemesini SSTable benzeri sıralı dosyalarda saklar.
- Arka planda LSM tarzı merge işlemi yapar.
- **PostgreSQL'in GIN index** türü de benzer şekilde postings list kullanır.

### 16.2 N-gram Index

Kelimelere bölmek yerine, tüm uzunluk-n alt dizeler (**n-gram**) index'lenir.

Örnek: `hello` kelimesinin tri-gram'ları (n=3): `hel`, `ell`, `llo`.

- En az 3 karakter uzunluğundaki **keyfi alt dizeler** aranabilir.
- Sorgu regex içerebilir.
- Dezavantaj: Index boyutu büyük.

### 16.3 Fuzzy Search (Bulanık Arama) — Levenshtein Automaton

Lucene, metin içinde belirli **edit distance** içindeki kelimeleri arayabilir (edit distance 1 = bir harf eklendi/silindi/değiştirildi).

**Yöntem:**
1. Term'ler bir **trie** benzeri **finite state automaton** olarak saklanır.
2. Sorgu için bir **Levenshtein automaton** oluşturulur: Verilen edit distance içindeki tüm kelimeleri tanıyan otomatik makine.
3. Bu sayede yazım hatalarına karşı toleranslı arama mümkün.

---
## 17. Vector Embeddings
### *Vektör Gömmeleri*

**Semantic search (Anlamsal arama):** Yalnızca eş anlamlılar veya yazım hatalarının ötesinde, belgenin **kavramlarını** ve kullanıcının **niyetini** anlamaya çalışır.

AI uygulamalarının önemli bir parçası:
- **Retrieval-Augmented Generation (RAG):** LLM'nin çıktısına arama sonuçlarını dahil etmek.
- Yardım sayfanızda "aboneliğinizi iptal etme" başlıklı bir sayfa varsa, kullanıcı "hesabımı nasıl kapatabilirim" veya "sözleşmeyi sonlandır" diye arama yaptığında da bulunsun.

### 17.1 Vector Embedding Nedir?

Bir belgenin **anlamını** anlamak için, **embedding model** (genellikle LLM) metin belgesini **float sayılarından oluşan bir vektöre** dönüştürür → **vector embedding**.

- Vektör, çok boyutlu bir uzayda **bir noktayı** temsil eder.
- Her float değeri, o boyut ekseninde belgenin konumunu gösterir.
- Anlamsal olarak benzer belgeler bu uzayda birbirine **yakın noktalar** üretir.

**Örnek (3 boyutlu):**
- Tarım → `[0.38, 0.83, 0.41]`
- Sebzeler → `[0.36, 0.64, 0.67]` (yakın!)
- Yıldız şeması → `[0.85, 0.10, -0.52]` (uzak)

Gerçekte embedding modelleri çok daha büyük vektörler kullanır (genellikle 1000+ boyut). Her sayının anlamını yorumlamaya çalışmıyoruz; sadece soyut bir çok boyutlu uzaydaki konumu.

**Mesafe fonksiyonları:**
- **Cosine similarity:** İki vektörün açısının kosinüsünü ölçer.
- **Euclidean distance:** İki nokta arasındaki düz çizgi mesafesi.

**Embedding modellerin gelişimi:**
- **Word2Vec, BERT, GPT:** Metin verisi için erken modeller, genellikle neural network.
- Video, ses, görüntü için embedding modeller de geliştirildi.
- **Multimodal modeller:** Tek model, metin ve görüntü gibi birden fazla modalite için embedding üretebilir.

### 17.2 Vector Indexes (Vektör İndeksleri)

Kullanıcı bir sorgu girdiğinde:
1. Sorgu metni embedding modele verilir → sorgunun vector embedding'i üretilir.
2. **Vector index**'te, sorgu vektörüne en yakın belge vektörleri aranır.

Yüksek boyutlu vektörlerde R-tree işe yaramaz. Özel vector index'ler kullanılır:

#### Flat Index
- Vektörler olduğu gibi index'te saklanır.
- Sorgu her vektörü okur ve mesafeyi ölçer.
- **Kesin** sonuç verir ama **yavaştır**.

#### IVF (Inverted File) Index
- Vektör uzayı cluster'lara (merkez noktalar = **centroid**) bölünür.
- Sorgu ile daha az vektör karşılaştırılır → daha hızlı.
- Yalnızca **yaklaşık** sonuç: Sorgu ve belge farklı partitionlara düşebilir.
- **Probe sayısı:** Kontrol edilecek partition sayısı. Daha fazla probe = daha doğru ama daha yavaş.

#### HNSW (Hierarchical Navigable Small World) Index
- Birden fazla **katman** içerir. Her katman bir **graph** olarak temsil edilir; node'lar vektörler, edge'ler yakınlık.
- Sorgu en üst katmandan başlar (az node) → en yakın node bulunur → bir alt katmana iner → daha yoğun bağlı, daha fazla node → en yakın node yeniden aranır → en alt katmana kadar devam eder.
- IVF gibi **yaklaşık** sonuç verir.

**Popüler implementasyonlar:**
- **Facebook Faiss:** IVF ve HNSW index'in çeşitli varyantları.
- **PostgreSQL pgvector:** Hem IVF hem de HNSW destekler.

In [8]:
# Cosine Similarity ve Euclidean Distance demo
import math

def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a**2 for a in v1))
    mag2 = math.sqrt(sum(b**2 for b in v2))
    return dot / (mag1 * mag2) if mag1 * mag2 != 0 else 0

def euclidean_distance(v1, v2):
    return math.sqrt(sum((a - b)**2 for a, b in zip(v1, v2)))

# 3 boyutlu örnek embedding'ler
agriculture = [0.38, 0.83, 0.41]
vegetables  = [0.36, 0.64, 0.67]
star_schema = [0.85, 0.10, -0.52]

# Sorgu: "Tarım" (agriculture vektörüne yakın)
query = agriculture

docs = {
    "Tarım (Agriculture)": agriculture,
    "Sebzeler (Vegetables)": vegetables,
    "Yıldız Şeması (Star Schema)": star_schema,
}

print("Query vektörü: Tarım (Agriculture)")
print("-" * 65)
print(f"{'Belge':<30} {'Cosine Sim':>12} {'Euclidean Dist':>16}")
print("-" * 65)
for name, vec in docs.items():
    cos = cosine_similarity(query, vec)
    euc = euclidean_distance(query, vec)
    print(f"{name:<30} {cos:>12.4f} {euc:>16.4f}")

print("\nYorum: Cosine similarity 1'e ne kadar yakınsa o kadar benzer.")
print("Euclidean distance ne kadar küçükse o kadar yakın.")

Query vektörü: Tarım (Agriculture)
-----------------------------------------------------------------
Belge                            Cosine Sim   Euclidean Dist
-----------------------------------------------------------------
Tarım (Agriculture)                  1.0000           0.0000
Sebzeler (Vegetables)                0.9477           0.3226
Yıldız Şeması (Star Schema)          0.1924           1.2723

Yorum: Cosine similarity 1'e ne kadar yakınsa o kadar benzer.
Euclidean distance ne kadar küçükse o kadar yakın.


---
## 18. Özet (Summary)

Bu bölümde veritabanlarının depolama ve erişimi nasıl gerçekleştirdiğinin temeline indik. Bir veri kaydedildiğinde ne olur? Daha sonra sorgulandığında veritabanı ne yapar?

### OLTP vs OLAP Storage Engines

**OLTP sistemleri:**
- Yüksek hacimli istek; her istek az sayıda kayıt okur/yazar, hızlı yanıt gerekir.
- Kayıtlara primary key veya secondary index üzerinden erişilir.
- Bu index'ler key'den record'a sıralı eşlemelerdir, range query'yi de destekler.

**Data warehouse ve analitik sistemler:**
- Çok sayıda kayıt üzerinde karmaşık okuma sorguları.
- Column-oriented storage + compression → Sorgular disk'ten minimum veri okur.
- JIT compilation veya vectorization → Minimum CPU zamanı harcanır.

### OLTP Tarafında İki Ana Yaklaşım

**1. Log-structured approach:**
- Dosyalara yalnızca append yapılır, yazılmış dosyalar asla güncellenmez, eski dosyalar silinir.
- Genel olarak yüksek write throughput sağlar.
- Bu gruba dahil: SSTables, LSM-trees, RocksDB, Cassandra, HBase, ScyllaDB, Lucene.

**2. Update-in-place approach:**
- Disk, üzerine yazılabilir sabit boyutlu page'ler seti olarak görülür.
- **B-tree**: Bu felsefenin en yaygın örneği; tüm büyük ilişkisel OLTP veritabanlarında ve pek çok ilişkisel olmayan veritabanında kullanılır.
- Kural olarak B-tree'ler okuma açısından daha iyi; log-structured storage'a göre daha yüksek read throughput ve daha düşük yanıt süresi.

### İleri Düzey Index'ler

- **Multidimensional index** (R-tree gibi): Enlem ve boylam gibi iki özelliğe aynı anda göre arama.
- **Full-text search index:** Aynı metinde birden fazla kelime aramayı inverted index ile destekler.
- **Vector index** (IVF, HNSW): Semantic search; belgeleri vector similarity ile karşılaştırır.

### Uygulama Geliştiricisi İçin Çıkarımlar

Storage engine'lerin iç işleyişini bilmek:
- Uygulamanıza en uygun aracı seçmenizi sağlar.
- Veritabanı tuning parametrelerini anlamlı şekilde yorumlamanıza olanak tanır.
- Seçtiğiniz veritabanının dokümantasyonunu çok daha iyi anlamanıza yardımcı olur.

---
### Bölümde Geçen Terimler Sözlüğü

| Terim | Açıklama |
|-------|----------|
| **Log** | Append-only kayıt dizisi (disk üzerinde) |
| **Index** | Veriyi hızlı bulmak için ek yapı |
| **SSTable** | Sorted String Table — key'e göre sıralı dosya |
| **Sparse index** | Yalnızca bazı key'leri içeren index |
| **Memtable** | LSM'de yazma için kullanılan in-memory sıralı yapı |
| **Compaction** | Segment'lerin merge edilip eski verilerin temizlenmesi |
| **Tombstone** | Silme işaretçisi |
| **LSM-tree** | Log-Structured Merge-tree |
| **B-tree** | Sabit boyutlu page'leri in-place güncelleyen ağaç |
| **WAL** | Write-Ahead Log — crash recovery için önceden yazılan log |
| **Bloom filter** | Varlık kontrolü için probabilistic veri yapısı |
| **False positive** | Olmayan bir şeyi varmış gibi raporlama |
| **Clustered index** | Veriyi doğrudan index içinde saklama |
| **Heap file** | Sırasız satır depolama |
| **Covering index** | Bazı sütunları index içinde saklayan index |
| **OLTP** | Online Transaction Processing |
| **OLAP** | Online Analytical Processing |
| **HTAP** | Hybrid Transactional/Analytical Processing |
| **Column-oriented storage** | Her sütunun değerlerini bir arada saklama |
| **Bitmap encoding** | Değerleri bit dizileriyle kodlama |
| **Run-length encoding** | Ardışık tekrarları sayarak sıkıştırma |
| **Write amplification** | Tek yazma isteğinin birden fazla disk I/O'suna dönüşmesi |
| **Inverted index** | Term → belge listesi eşleşmesi |
| **Postings list** | Bir terimi içeren belgelerin listesi |
| **N-gram** | Uzunluğu n olan ardışık karakter dizisi |
| **Edit distance** | Bir string'den diğerine geçmek için gereken değişiklik sayısı |
| **Vector embedding** | Metni float dizisi olarak temsil etme |
| **Semantic search** | Anlamsal benzerliğe dayalı arama |
| **RAG** | Retrieval-Augmented Generation |
| **HNSW** | Hierarchical Navigable Small World |
| **IVF** | Inverted File (vector index) |
| **Data cube** | Önceden hesaplanmış çok boyutlu agregat |
| **Materialized view** | Sorgu sonuçlarının diske yazılmış kopyası |